<a href="https://colab.research.google.com/github/epicdata2git/diarization/blob/main/%ED%97%88%EC%9E%AC%EC%A0%95_2426572_006_Speaker_Diarization_Separation_STT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

* 동영상 수집
1)

1. 초등학생의 토론 레전드 배틀 (https://www.youtube.com/shorts/sNhjvNjvUmo)
2. 파일을 MP4로 다운로드 https://en1.savefrom.net/13RZ/
3. Convertio (https://convertio.co/kr/mp4-wav/) 통해서 WAV파일로 변형
4. 깃허브 repository에 업로드 (https://github.com/epicdata2git/diarization)

#### 화자 구분 (Speaker Diarization)


#### 모델 링크 : https://huggingface.co/pyannote/speaker-diarization-3.1


In [4]:
!pip install pyannote.audio

!pip install transformers
!pip install datasets

!pip install accelerate

In [6]:
# 불필요한 Warning 제거
from transformers.utils import logging
logging.set_verbosity_error()

In [9]:
from getpass import getpass

HF_TOKEN = getpass("Hugging Face API Token : ")

Hugging Face API Token : ··········


In [10]:
from huggingface_hub import login

# Hugging Face에 로그인
login(token=HF_TOKEN)

#### wave 화일 upload

#### pipeline 생성


In [11]:
# instantiate the pipeline
from pyannote.audio import Pipeline
pipeline = Pipeline.from_pretrained(
  "pyannote/speaker-diarization-3.1",
  use_auth_token=HF_TOKEN)




DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for _speechbrain_save
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint load hook for _speechbrain_load
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for save
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint load hook for load
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for _save
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint load hook for _recover


In [12]:
# # model 출력

# print("pipeline.model == ", pipeline)

In [13]:
import torch

if torch.cuda.is_available():
    print("GPU is available")
    pipeline.to(torch.device("cuda"))
else:
    print("GPU is not available")



GPU is not available


In [14]:
# Wave 파일 download
#
import urllib.request
import os

# f_audio =  "trumpclinton-cut.wav"
# f_audio = "simahn-cut.wav"
f_audio = "galaxyiphone_battle.wav"

# 이미지 다운로드
# Construct the raw file URL using the correct format
url_audio = f"https://raw.githubusercontent.com/epicdata2git/diarization/main/{f_audio}"
dir_audio = "./"

path_audio = os.path.join(dir_audio, f_audio)
urllib.request.urlretrieve(url_audio, path_audio) # 이미지 다운로드

('./galaxyiphone_battle.wav', <http.client.HTTPMessage at 0x7fa25cb14a10>)

In [15]:
# run the pipeline on an audio file

# # f_audio = "./trumpclinton-cut.wav"
# f_audio = "./simahn-cut.wav"


diarization = pipeline(path_audio)

/usr/local/lib/python3.11/dist-packages/pyannote/audio/models/blocks/pooling.py:104: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1831.)
  std = sequences.std(dim=-1, correction=1)


In [16]:
#  dump the diarization output to disk using RTTM format

f_rttm = f_audio +".rttm"

with open(f_rttm, "w") as rttm:
    diarization.write_rttm(rttm)

In [17]:
with open(f_rttm, 'r') as f:
    for line in f:
        print(line, end='')

SPEAKER galaxyiphone_battle 1 0.031 10.058 <NA> <NA> SPEAKER_01 <NA> <NA>
SPEAKER galaxyiphone_battle 1 6.578 0.034 <NA> <NA> SPEAKER_02 <NA> <NA>
SPEAKER galaxyiphone_battle 1 8.924 0.034 <NA> <NA> SPEAKER_02 <NA> <NA>
SPEAKER galaxyiphone_battle 1 8.958 1.418 <NA> <NA> SPEAKER_03 <NA> <NA>
SPEAKER galaxyiphone_battle 1 10.375 2.531 <NA> <NA> SPEAKER_01 <NA> <NA>
SPEAKER galaxyiphone_battle 1 12.907 1.974 <NA> <NA> SPEAKER_03 <NA> <NA>
SPEAKER galaxyiphone_battle 1 14.881 6.362 <NA> <NA> SPEAKER_01 <NA> <NA>
SPEAKER galaxyiphone_battle 1 14.948 0.270 <NA> <NA> SPEAKER_03 <NA> <NA>
SPEAKER galaxyiphone_battle 1 21.547 2.008 <NA> <NA> SPEAKER_02 <NA> <NA>
SPEAKER galaxyiphone_battle 1 23.555 0.051 <NA> <NA> SPEAKER_00 <NA> <NA>
SPEAKER galaxyiphone_battle 1 23.605 0.017 <NA> <NA> SPEAKER_02 <NA> <NA>
SPEAKER galaxyiphone_battle 1 23.622 0.034 <NA> <NA> SPEAKER_01 <NA> <NA>
SPEAKER galaxyiphone_battle 1 23.622 2.076 <NA> <NA> SPEAKER_00 <NA> <NA>
SPEAKER galaxyiphone_battle 1 23.656 2.37

### RTTM 이용한 wav 분리

In [18]:
import wave
import contextlib
import os

In [19]:

# RTTM 파일 읽기 함수
def read_rttm(rttm_file):
    speaker_segments = {}
    with open(rttm_file, 'r') as file:
        for line in file:
            parts = line.strip().split()
            if len(parts) < 8 or parts[0] != 'SPEAKER':
                continue
            speaker = parts[7]  # 화자 ID
            start_time = float(parts[3])  # 시작 시간 (초)
            duration = float(parts[4])  # 지속 시간 (초)
            end_time = start_time + duration

            if speaker not in speaker_segments:
                speaker_segments[speaker] = []
            speaker_segments[speaker].append((start_time, end_time))
    return speaker_segments

In [20]:
# 화자별 오디오 분리 함수
def split_audio_by_speaker(audio_file, rttm_file, output_dir):
    # 오디오 파일 정보 읽기
    with contextlib.closing(wave.open(audio_file, 'rb')) as audio:
        params = audio.getparams()
        framerate = params.framerate  # 샘플링 속도
        n_channels = params.nchannels  # 채널 수
        sampwidth = params.sampwidth  # 샘플 폭 (바이트)
        n_frames = params.nframes  # 총 프레임 수

        audio_data = audio.readframes(n_frames)  # 원본 오디오 데이터

    # RTTM 파일에서 화자 구간 읽기
    speaker_segments = read_rttm(rttm_file)

    # 출력 디렉토리 생성
    os.makedirs(output_dir, exist_ok=True)

    # 화자별로 오디오 분리 및 저장
    for speaker, segments in speaker_segments.items():
        speaker_audio_data = bytearray()

        for start, end in segments:
            start_frame = int(start * framerate)
            end_frame = int(end * framerate)

            segment_data = audio_data[start_frame * sampwidth * n_channels : end_frame * sampwidth * n_channels]
            speaker_audio_data.extend(segment_data)

        # 화자별 오디오 저장
        output_path = os.path.join(output_dir, f"{speaker}.wav")
        with wave.open(output_path, 'wb') as out_audio:
            out_audio.setnchannels(n_channels)
            out_audio.setsampwidth(sampwidth)
            out_audio.setframerate(framerate)
            out_audio.writeframes(speaker_audio_data)

        print(f"Saved {speaker} audio to {output_path}")

In [21]:
# 샘플 실행

str_output = f_audio + "speakers"
split_audio_by_speaker(f_audio, f_rttm, str_output)

Saved SPEAKER_01 audio to galaxyiphone_battle.wavspeakers/SPEAKER_01.wav
Saved SPEAKER_02 audio to galaxyiphone_battle.wavspeakers/SPEAKER_02.wav
Saved SPEAKER_03 audio to galaxyiphone_battle.wavspeakers/SPEAKER_03.wav
Saved SPEAKER_00 audio to galaxyiphone_battle.wavspeakers/SPEAKER_00.wav


#### 화자 분할 (Speaker Seperation)

In [22]:
from speechbrain.pretrained import SepformerSeparation as separator

<ipython-input-22-1591e43229fe>:1: UserWarning: Module 'speechbrain.pretrained' was deprecated, redirecting to 'speechbrain.inference'. Please update your script. This is a change from SpeechBrain 1.0. See: https://github.com/speechbrain/speechbrain/releases/tag/v1.0.0
  from speechbrain.pretrained import SepformerSeparation as separator


In [23]:
# Conv-TasNet으로 중첩된 음성 분리

separator = separator.from_hparams(source="speechbrain/sepformer-wham", savedir="tmpdir")
import torch

if torch.cuda.is_available():
    print("GPU is available")
    separator.to(torch.device("cuda"))
else:
    print("GPU is not available")


separator.to("cpu")

INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Using symlink found at '/content/tmpdir/hyperparams.yaml'
INFO:speechbrain.utils.fetching:Fetch custom.py: Fetching from HuggingFace Hub 'speechbrain/sepformer-wham' if not cached
DEBUG:speechbrain.utils.parameter_transfer:Collecting files (or symlinks) for pretraining in tmpdir.
INFO:speechbrain.utils.fetching:Fetch masknet.ckpt: Using symlink found at '/content/tmpdir/masknet.ckpt'
DEBUG:speechbrain.utils.parameter_transfer:Set local path in self.paths["masknet"] = /content/tmpdir/masknet.ckpt
INFO:speechbrain.utils.fetching:Fetch encoder.ckpt: Using symlink found at '/content/tmpdir/encoder.ckpt'
DEBUG:speechbrain.utils.parameter_transfer:Set local path in self.paths["encoder"] = /content/tmpdir/encoder.ckpt
INFO:speechbrain.utils.fetching:Fetch decoder.ckpt: Using symlink found at '/content/tmpdir/decoder.ckpt'
DEBUG:speechbrain.utils.parameter_transfer:Set local path in self.paths["decoder"] = /content/tmpdir/decoder.ckpt
INF

GPU is not available


SepformerSeparation(
  (mods): ModuleDict(
    (encoder): Encoder(
      (conv1d): Conv1d(1, 256, kernel_size=(16,), stride=(8,), bias=False)
    )
    (decoder): Decoder(256, 1, kernel_size=(16,), stride=(8,), bias=False)
    (masknet): Dual_Path_Model(
      (norm): GroupNorm(1, 256, eps=1e-08, affine=True)
      (conv1d): Conv1d(256, 256, kernel_size=(1,), stride=(1,), bias=False)
      (dual_mdl): ModuleList(
        (0-1): 2 x Dual_Computation_Block(
          (intra_mdl): SBTransformerBlock(
            (mdl): TransformerEncoder(
              (layers): ModuleList(
                (0-7): 8 x TransformerEncoderLayer(
                  (self_att): MultiheadAttention(
                    (att): MultiheadAttention(
                      (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
                    )
                  )
                  (pos_ffn): PositionalwiseFeedForward(
                    (ffn): Sequential(
                      (0

In [24]:
f_audio="galaxyiphone_battle.wav"

separated_audio = separator.separate_file(path=f_audio)

Resampling the audio from 44100 Hz to 8000 Hz


In [36]:
import torchaudio

torchaudio.save("source1hat.wav", separated_audio[:, :, 0].detach().cpu(), 8000)
torchaudio.save("source2hat.wav", separated_audio[:, :, 1].detach().cpu(), 8000)

#### 결과가 왜 좋지 않은가??  ==> 모델 카드 확인하기

In [25]:
# !pip install pydub

In [38]:
# # 각 화자의 오디오 저장
# import numpy as np
# from pydub import AudioSegment

# for i, speaker_audio in enumerate(separated_audio):
#     # PyTorch Tensor를 NumPy 배열로 변환
#     speaker_audio_np = speaker_audio.numpy()

#     # NumPy 배열을 AudioSegment로 변환
#     audio_segment = AudioSegment(
#         speaker_audio_np.tobytes(),  # NumPy 배열 데이터를 바이트로 변환
#         frame_rate=8000,            # 샘플링 속도 (모델에 따라 다름, sepformer는 8kHz)
#         sample_width=2,             # 샘플 폭 (16비트 PCM = 2바이트)
#         channels=1                  # 모노 채널
#     )

#     # 화자별 오디오 저장
#     output_path = f"speaker_{i}.wav"
#     audio_segment.export(output_path, format="wav")
#     print(f"Saved speaker {i} audio to {output_path}")

\### 한국어가 지원되는 화자 분리 라이브러리



####

In [26]:
# Workaround

import locale
locale.getpreferredencoding = lambda: "UTF-8"

In [27]:
!pip install asteroid

In [ ]:
from asteroid.models import ConvTasNet
import torchaudio
import os

# 1. 사전 학습된 ConvTasNet 모델 로드
model = ConvTasNet.from_pretrained("JorisCos/ConvTasNet_Libri2Mix_sepclean_16k")

if torch.cuda.is_available():
    print("GPU is available")
    model.to(torch.device("cuda"))
else:
    print("GPU is not available")

# 2. 중첩된 오디오 파일 로드
# Define f_audio here
f_audio = "galaxyiphone_battle.wav"  # or the actual path to your audio file
input_audio_path = f_audio  # 중첩된 음성 파일 경로
waveform, sample_rate = torchaudio.load(input_audio_path)

# 3. 샘플링 속도 확인 및 변환 (16kHz로 변환 필요 시)
if sample_rate != 16000:
    resample = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)
    waveform = resample(waveform)
    sample_rate = 16000

# 4. 모델에 입력할 텐서 형식으로 변환
# Convert stereo to mono by averaging the two channels
waveform = waveform.mean(0, keepdim=True)  # Average across channels to get mono

# 배치 차원 추가 (batch_size, num_channels, num_samples)
waveform = waveform.unsqueeze(0)

# 5. 화자 분리 수행
with torch.no_grad():
    separated_sources = model.separate(waveform)  # (batch_size, num_sources, num_samples)

# 6. 분리된 음성 저장
output_dir = "separated_speakers"
os.makedirs(output_dir, exist_ok=True)

f_audio_names = []
for i, source in enumerate(separated_sources[0]):  # batch 차원 제거
    output_path = f"{output_dir}/speaker_{i}.wav"
    torchaudio.save(output_path, source.unsqueeze(0), sample_rate=16000)
    f_audio_names.append(output_path)
    print(f"Saved speaker {i} audio to {output_path}")

GPU is not available


In [5]:
# 음성 PLAY (Iphone_girl)
from IPython.display import Audio

# Pass the filename as a string to the Audio function
Audio("separated_speakers/speaker_0.wav")

In [6]:
# 음성 PLAY (galaxy_boy)
Audio("separated_speakers/speaker_1.wav")

#### Audio에서 Text 추출

In [3]:
# speaker_1.wav 파일 경로 설정 (Galaxy 사용 소년)
audio_file = "separated_speakers/speaker_1.wav"

# 음성 파일을 텍스트로 변환
segments, info = pipeline.transcribe(audio_file,  language="ko", beam_size=5)  # 한국어로 설정

# 변환된 텍스트 출력
for segment in segments:
    print(f"[{segment.start:.2f}s -> {segment.end:.2f}s] {segment.text}")

[0.00s -> 2.76s]  최근에 나온 S23 보셨나요?
[3.76s -> 5.90s]  일단 그게 100배 줌이 가능합니다
[5.90s -> 7.64s]  그러나 아이폰
[7.64s -> 9.00s]  줌하면 깨져요
[9.00s -> 11.24s]  화질이 깨진다고
[11.24s -> 12.94s]  카메라 화질이 깨진다고요
[14.94s -> 16.04s]  아이폰 개비싸죠?
[16.04s -> 17.00s]  갤럭시 개꿀이죠?
[17.00s -> 19.44s]  갤럭시 휴대폰이죠?
[19.44s -> 21.34s]  응? 갤럭시는 우리 아빠가 사준거죠?
[23.86s -> 25.36s]  우리 엄마 할머니 할아버지
[28.60s -> 29.40s]  비싸면 왜 사요?
[29.40s -> 31.24s]  비싼데 성분까지 안 좋아
[32.10s -> 32.74s]  누가 사요?
[38.64s -> 40.30s]  그럼 갤럭시가 안 좋다고 생각하시나요?
[41.76s -> 43.04s]  아이폰이 더 안 좋아요?
[43.04s -> 43.90s]  아이폰이 더 안 좋아요?
[43.90s -> 45.60s]  콘서트장 가봤어요? 아이폰 가지고?
[46.80s -> 48.20s]  그니까 아이폰 가지고 뭘 해요
[50.84s -> 52.70s]  구글 기프트카드 아이폰으로 못 쓰죠?
[55.60s -> 56.60s]  오빠한테 돈이요?
[56.60s -> 57.66s]  그걸 왜
[57.66s -> 59.36s]  그걸 왜 갤럭시한테 물어봐요?


KeyboardInterrupt: 

In [ ]:
# speaker_0.wav 파일 경로 설정 (iphone 사용 소녀)
audio_file = "separated_speakers/speaker_0.wav"

# 음성 파일을 텍스트로 변환
segments, info = pipeline.transcribe(audio_file,  language="ko", beam_size=5)  # 한국어로 설정

# 변환된 텍스트 출력
for segment in segments:
    print(f"[{segment.start:.2f}s -> {segment.end:.2f}s] {segment.text}")

###  연습문제
#### Systran/faster-whisper-large-v3 모델을 사용하여
#### 분할된 음성에서 text 추출하기

#### 화자 구분 (Speaker Diarization)
#### 모델 링크 : https://huggingface.co/Systran/faster-whisper-large-v3

In [ ]:
!pip install pyannote.audio

!pip install transformers
!pip install datasets

!pip install accelerate

In [4]:
# 불필요한 Warning 제거
from transformers.utils import logging
logging.set_verbosity_error()

In [ ]:
from getpass import getpass

HF_TOKEN = getpass("Hugging Face API Token : ")

In [6]:
from huggingface_hub import login

# Hugging Face에 로그인
login(token=HF_TOKEN)

#### wav 파일 업로드
#### 파이프라인 생성

In [1]:
!pip install faster-whisper
import torch  # Import torch here
from faster_whisper import WhisperModel

pipeline = WhisperModel("large-v3", device="cuda" if torch.cuda.is_available() else "cpu")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
import torch

if torch.cuda.is_available():
    print("GPU is available")
    pipeline.to(torch.device("cuda"))
else:
    print("GPU is not available")

In [ ]:
# Wave 파일 download
#
import urllib.request
import os

# f_audio =  "trumpclinton-cut.wav"
# f_audio = "simahn-cut.wav"
f_audio = "galaxyiphone_battle.wav"

# 이미지 다운로드
# Construct the raw file URL using the correct format
url_audio = f"https://raw.githubusercontent.com/epicdata2git/diarization/main/{f_audio}"
dir_audio = "./"

path_audio = os.path.join(dir_audio, f_audio)
urllib.request.urlretrieve(url_audio, path_audio) # 이미지 다운로드b

In [ ]:
# run the pipeline on an audio file

# # f_audio = "./trumpclinton-cut.wav"
# f_audio = "./simahn-cut.wav"
quoted_filename = urllib.parse.quote(f_audio)

diarization = pipeline(path_audio)

In [ ]:
#  dump the diarization output to disk using RTTM format

f_rttm = f_audio +".rttm"

with open(f_rttm, "w") as rttm:
    diarization.write_rttm(rttm)

In [ ]:
with open(f_rttm, 'r') as f:
    for line in f:
        print(line, end='')

#### RTTM 이용한 WAV 분리

In [9]:
import wave
import contextlib
import os

In [ ]:

# RTTM 파일 읽기 함수
def read_rttm(rttm_file):
    speaker_segments = {}
    with open(rttm_file, 'r') as file:
        for line in file:
            parts = line.strip().split()
            if len(parts) < 8 or parts[0] != 'SPEAKER':
                continue
            speaker = parts[7]  # 화자 ID
            start_time = float(parts[3])  # 시작 시간 (초)
            duration = float(parts[4])  # 지속 시간 (초)
            end_time = start_time + duration

            if speaker not in speaker_segments:
                speaker_segments[speaker] = []
            speaker_segments[speaker].append((start_time, end_time))
    return speaker_segments

In [ ]:
# 화자별 오디오 분리 함수
def split_audio_by_speaker(audio_file, rttm_file, output_dir):
    # 오디오 파일 정보 읽기
    with contextlib.closing(wave.open(audio_file, 'rb')) as audio:
        params = audio.getparams()
        framerate = params.framerate  # 샘플링 속도
        n_channels = params.nchannels  # 채널 수
        sampwidth = params.sampwidth  # 샘플 폭 (바이트)
        n_frames = params.nframes  # 총 프레임 수

        audio_data = audio.readframes(n_frames)  # 원본 오디오 데이터

    # RTTM 파일에서 화자 구간 읽기
    speaker_segments = read_rttm(rttm_file)

    # 출력 디렉토리 생성
    os.makedirs(output_dir, exist_ok=True)

    # 화자별로 오디오 분리 및 저장
    for speaker, segments in speaker_segments.items():
        speaker_audio_data = bytearray()

        for start, end in segments:
            start_frame = int(start * framerate)
            end_frame = int(end * framerate)

            segment_data = audio_data[start_frame * sampwidth * n_channels : end_frame * sampwidth * n_channels]
            speaker_audio_data.extend(segment_data)

        # 화자별 오디오 저장
        output_path = os.path.join(output_dir, f"{speaker}.wav")
        with wave.open(output_path, 'wb') as out_audio:
            out_audio.setnchannels(n_channels)
            out_audio.setsampwidth(sampwidth)
            out_audio.setframerate(framerate)
            out_audio.writeframes(speaker_audio_data)

        print(f"Saved {speaker} audio to {output_path}")

In [ ]:
# 샘플 실행

str_output = f_audio + "speakers"
split_audio_by_speaker(f_audio, f_rttm, str_output)

#### 화자 분할 (Speaker Seperation)

In [ ]:
from speechbrain.pretrained import SepformerSeparation as separator

In [ ]:
# Conv-TasNet으로 중첩된 음성 분리

separator = separator.from_hparams(source="speechbrain/sepformer-wham", savedir="tmpdir")
import torch

if torch.cuda.is_available():
    print("GPU is available")
    separator.to(torch.device("cuda"))
else:
    print("GPU is not available")


separator.to("cpu")

In [ ]:
f_audio="galaxyiphone_battle.wav"

separated_audio = separator.separate_file(path=f_audio)

In [ ]:
import torchaudio

torchaudio.save("source1hat.wav", separated_audio[:, :, 0].detach().cpu(), 8000)
torchaudio.save("source2hat.wav", separated_audio[:, :, 1].detach().cpu(), 8000)

#### 한국어가 지원되는 화자 분리 라이브러리

In [ ]:
# Workaround

import locale
locale.getpreferredencoding = lambda: "UTF-8"

In [ ]:
!pip install asteroid

In [ ]:
from asteroid.models import ConvTasNet
import torchaudio
import os

# 1. 사전 학습된 ConvTasNet 모델 로드
model = ConvTasNet.from_pretrained("JorisCos/ConvTasNet_Libri2Mix_sepclean_16k")

if torch.cuda.is_available():
    print("GPU is available")
    model.to(torch.device("cuda"))
else:
    print("GPU is not available")

# 2. 중첩된 오디오 파일 로드
# Define f_audio here
f_audio = "galaxyiphone_battle.wav"  # or the actual path to your audio file
input_audio_path = f_audio  # 중첩된 음성 파일 경로
waveform, sample_rate = torchaudio.load(input_audio_path)

# 3. 샘플링 속도 확인 및 변환 (16kHz로 변환 필요 시)
if sample_rate != 16000:
    resample = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)
    waveform = resample(waveform)
    sample_rate = 16000

# 4. 모델에 입력할 텐서 형식으로 변환
# Convert stereo to mono by averaging the two channels
waveform = waveform.mean(0, keepdim=True)  # Average across channels to get mono

# 배치 차원 추가 (batch_size, num_channels, num_samples)
waveform = waveform.unsqueeze(0)

# 5. 화자 분리 수행
with torch.no_grad():
    separated_sources = model.separate(waveform)  # (batch_size, num_sources, num_samples)

# 6. 분리된 음성 저장
output_dir = "separated_speakers"
os.makedirs(output_dir, exist_ok=True)

f_audio_names = []
for i, source in enumerate(separated_sources[0]):  # batch 차원 제거
    output_path = f"{output_dir}/speaker_{i}.wav"
    torchaudio.save(output_path, source.unsqueeze(0), sample_rate=16000)
    f_audio_names.append(output_path)
    print(f"Saved speaker {i} audio to {output_path}")

In [ ]:
# 음성 PLAY (Iphone_girl)
from IPython.display import Audio

# Pass the filename as a string to the Audio function
Audio("separated_speakers/speaker_0.wav")

In [ ]:
# 음성 PLAY (galaxy_boy)
Audio("separated_speakers/speaker_1.wav")